SETUP THE NWB DATA

In [2]:
import sys
sys.path.insert(0, '/code/src')

import glob
import os
import numpy as np
import pandas as pd
import seaborn as sns
import pynwb
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from datetime import datetime
from scipy.stats import linregress

sns.set_theme(context='talk', style='ticks', palette='colorblind')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.titlesize'] = 'medium'
plt.rcParams['axes.titlelocation'] = 'left'

sns.set_context('talk')
pd.set_option('display.max_columns', None)

DATA_ROOT = '/data/dynamicrouting_datacube'

SESSION_NUMBER = 1        # <-- change this to switch session

# every session available on disk, sorted so the numbering is stable
session_paths = sorted(glob.glob('%s/*/*.nwb.zarr' % DATA_ROOT))
session_ids = [os.path.basename(p).replace('.nwb.zarr', '') for p in session_paths]
print('%d sessions available' % len(session_ids))
for position, available_id in enumerate(session_ids):
    print('  %2d  %s' % (position, available_id))

session_id = session_ids[SESSION_NUMBER]
nwb_path = session_paths[SESSION_NUMBER]

# cache: re-running this cell for the same session does not reload the file
if 'session_cache' not in globals():
    session_cache = {}
if session_id not in session_cache:
    print('\nloading %s ...' % session_id)
    session_cache[session_id] = pynwb.read_nwb(nwb_path)
else:
    print('\nusing cached %s' % session_id)

session = session_cache[session_id]
trials = session.trials[:]
print('session %d: %s, %d trials' % (SESSION_NUMBER, session_id, len(trials)))

12 sessions available
   0  662892_2023-08-24
   1  664851_2023-11-16
   2  667252_2023-09-28
   3  708016_2024-04-29
   4  712815_2024-05-22
   5  713655_2024-08-09
   6  714748_2024-06-24
   7  715710_2024-07-16
   8  741137_2024-10-10
   9  742903_2024-10-23
  10  743199_2024-12-05
  11  759434_2025-02-04

loading 664851_2023-11-16 ...


/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_front_camera/data' (400638, 500) float32 read-only>
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_side_camera/data' (400680, 500) float32 read-only>
  warnings.warn(


session 1: 664851_2023-11-16, 527 trials


SETUP THE METADATA

In [3]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = 'api.allenneuraldynamics.org'
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(host=API_GATEWAY_HOST, version='v2',
                                    database=DATABASE, collection=COLLECTION)

aggregate = [
    {'$match': {
        'data_description.project_name': 'Dynamic Routing',
        'data_description.data_level': 'derived',
        'processing.data_processes': {'$elemMatch': {
            'process_type': 'File format conversion',
            'start_date_time': {'$regex': '^2026-08-04'}}}}},
    {'$project': {
        'name': 1,
        'subject_id': '$data_description.subject_id',
        'genotype': '$subject.subject_details.genotype',
        'date_of_birth': '$subject.subject_details.date_of_birth',
        'sex': '$subject.subject_details.sex',
        'session_start_time': '$acquisition.acquisition_start_time',
        'session_end_time': '$acquisition.acquisition_end_time',
        'stimulus_epochs': '$acquisition.stimulus_epochs',
        'project_name': '$data_description.project_name',
        'modality': '$data_description.modalities.name',
        'targeted_structure': ('$acquisition.data_streams.configurations.probes'
                               '.primary_targeted_structure.name')}},
]

records = docdb_api_client.aggregate_docdb_records(pipeline=aggregate)
for r in records:
    dr = next((e for e in r.get('stimulus_epochs', [])
               if e.get('stimulus_name') == 'DynamicRouting1'), None)
    r['dr_performance'] = dr['performance_metrics'] if dr else None

metadata_df = pd.DataFrame(records)
metadata_df['trials_total'] = metadata_df['dr_performance'].apply(
    lambda x: x['trials_total'] if x else None)
metadata_df['trials_rewarded'] = metadata_df['dr_performance'].apply(
    lambda x: x['trials_rewarded'] if x else None)

# independent cross-check: these counts come from the metadata index, not the NWB file
this_session_row = metadata_df[metadata_df.name.str.contains(session_id, na=False)]
print('Metadata trials_total: %s, NWB trials: %d'
      % (this_session_row.trials_total.values, len(trials)))
print('Metadata trials_rewarded: %s, NWB is_rewarded: %d'
      % (this_session_row.trials_rewarded.values, trials.is_rewarded.sum()))
metadata_df.head(3)

Metadata trials_total: [527], NWB trials: 527
Metadata trials_rewarded: [139], NWB is_rewarded: 145


,_id,name,subject_id,genotype,date_of_birth,sex,session_start_time,session_end_time,stimulus_epochs,project_name,modality,targeted_structure,dr_performance,trials_total,trials_rewarded
0,c5bc8e0e-81ee-4e52-a7b6-90089b0dfc5f,ecephys_742903_2024-10-23_14-12-23_nwb_2026-08...,742903,Vip-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2024-05-16,Female,2024-10-23 14:12:23-07:00,2024-10-23 16:15:54.652646-07:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Primary somatosensory area], [Caudoputamen]...","{'object_type': 'Performance metrics', 'output...",489,124
1,7f603b00-8b9f-4a66-8b9d-1d9ed27b40d4,ecephys_743199_2024-12-05_12-42-34_nwb_2026-08...,743199,VGAT-ChR2-YFP/wt,2024-05-18,Female,2024-12-05 12:42:34-08:00,2024-12-05 14:39:42.445214-08:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Periaqueductal gray], [Red nucleus], [Red n...","{'object_type': 'Performance metrics', 'output...",490,132
2,66dc0f20-45dc-4a65-ac3a-0a04ac0e1df4,ecephys_662892_2023-08-24_14-28-28_nwb_2026-08...,662892,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,2022-12-24,Female,2023-08-24 14:28:28-07:00,2023-08-24 16:28:14.588745-07:00,"[{'object_type': 'Stimulus epoch', 'stimulus_s...",Dynamic Routing,"[Extracellular electrophysiology, Behavior, Be...","[[[Piriform area], [root]]]","{'object_type': 'Performance metrics', 'output...",476,123


UNITS QUALITY CONTROL

In [4]:
units_table = session.units[:]
print('This session has %d units before quality control' % len(units_table))

# QC thresholds
MAX_ISI_VIOLATIONS_RATIO = 0.5    # lower is better
MAX_AMPLITUDE_CUTOFF = 0.1        # lower is better
MIN_PRESENCE_RATIO = 0.95         # higher is better

good_units = units_table[
    (units_table.isi_violations_ratio < MAX_ISI_VIOLATIONS_RATIO) &
    (units_table.amplitude_cutoff < MAX_AMPLITUDE_CUTOFF) &
    (units_table.presence_ratio > MIN_PRESENCE_RATIO)
]

this_structure_units_table = good_units[good_units.structure == 'MOs']
print('%d good units in MOs' % len(this_structure_units_table))

This session has 2210 units before quality control
169 good units in MOs


TRIAL STRUCTURE

In [5]:
# Trial structure
print('Trial structure: go %d + nogo %d + catch %d = %d, total trials %d'
      % (trials.is_go.sum(), trials.is_nogo.sum(), trials.is_catch.sum(),
         trials.is_go.sum() + trials.is_nogo.sum() + trials.is_catch.sum(),
         len(trials)))
print('hit %d + miss %d = go %d'
      % (trials.is_hit.sum(), trials.is_miss.sum(), trials.is_go.sum()))
print('false alarm %d + correct reject %d = nogo %d'
      % (trials.is_false_alarm.sum(), trials.is_correct_reject.sum(),
         trials.is_nogo.sum()))

# Responses types
n_catch_response = (trials.is_catch & trials.is_response).sum()
print('\nhit %d + false alarm %d + catch response %d = %d, is_response %d'
      % (trials.is_hit.sum(), trials.is_false_alarm.sum(), n_catch_response,
         trials.is_hit.sum() + trials.is_false_alarm.sum() + n_catch_response,
         trials.is_response.sum()))

# Response window
window_start_offset = trials.response_window_start_time - trials.stim_start_time
window_stop_offset = trials.response_window_stop_time - trials.stim_start_time
print('\nresponse window start offset: %.4f to %.4f s (median %.4f)'
      % (window_start_offset.min(), window_start_offset.max(),
         window_start_offset.median()))
print('response window stop  offset: %.4f to %.4f s (median %.4f)'
      % (window_stop_offset.min(), window_stop_offset.max(),
         window_stop_offset.median()))

# Save outcome trial structure
trials['outcome'] = np.select(
    [trials.is_hit, trials.is_miss, trials.is_false_alarm, trials.is_correct_reject],
    ['hit', 'miss', 'false alarm', 'correct reject'], default='unscored')

Trial structure: go 145 + nogo 342 + catch 40 = 527, total trials 527
hit 139 + miss 6 = go 145
false alarm 53 + correct reject 289 = nogo 342

hit 139 + false alarm 53 + catch response 1 = 193, is_response 193

response window start offset: 0.0558 to 0.0763 s (median 0.0560)
response window stop  offset: 0.9732 to 1.0103 s (median 0.9901)


SETUP NEURAL ACTIVITY DURING QUIESCENT PERIOD

In [6]:
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

QUIESCENT_WINDOW_S = 1.5      # pre-stimulus window: state before the trial
DECODE_TARGET = 'consecutive_unrewarded'
N_FOLDS = 5


def get_binned_triggered_spike_counts_fast(spike_times, stim_times, bins):
    """Workshop helper: spike counts per trial per bin, via searchsorted."""
    spike_times = np.asarray(spike_times)
    stim_times = np.asarray(stim_times)
    bins = np.asarray(bins)
    counts = np.zeros((stim_times.size, bins.size - 1), dtype=int)
    for i, stim in enumerate(stim_times):
        counts[i, :] = np.diff(np.searchsorted(spike_times, stim + bins, side='left'))
    return counts


# one spike count per neuron per trial, in the quiescent window before onset
window_bins = np.array([-QUIESCENT_WINDOW_S, 0.0])
stim_onset_array = trials.stim_start_time.values

neural_count_array = np.zeros((len(trials), len(this_structure_units_table)))
for column_index, spike_time_array in enumerate(
        this_structure_units_table.spike_times.values):
    neural_count_array[:, column_index] = get_binned_triggered_spike_counts_fast(
        spike_time_array, stim_onset_array, window_bins).ravel()

print('neural matrix: %d trials x %d MOs units' % neural_count_array.shape)
print('mean rate in window: %.2f Hz per neuron'
      % (neural_count_array.mean() / QUIESCENT_WINDOW_S))
print('dead units in this window (zero spikes on every trial): %d'
      % int((neural_count_array.sum(axis=0) == 0).sum()))

neural matrix: 527 trials x 169 MOs units
mean rate in window: 3.40 Hz per neuron
dead units in this window (zero spikes on every trial): 0


In [8]:
def trailing_streak(flag_array):
    """Number of consecutive True values immediately preceding each position."""
    streak = np.zeros(len(flag_array), dtype=int)
    run_length = 0
    for position, flag in enumerate(flag_array):
        streak[position] = run_length
        run_length = run_length + 1 if flag else 0
    return streak


# streaks over the full trial sequence: contiguous trials, nothing skipped
trials['consecutive_rewarded'] = trailing_streak(trials.is_rewarded.values)
trials['consecutive_unrewarded'] = trailing_streak(~trials.is_rewarded.values)

In [9]:
# trials entering the decode: exclude catch, require a defined target
decode_mask = (~trials.is_catch & trials[DECODE_TARGET].notna()).values
X_neural = neural_count_array[decode_mask]
y_target = trials.loc[decode_mask, DECODE_TARGET].values.astype(float)
trial_position = np.flatnonzero(decode_mask)

print('%d trials, target = %s (range %d-%d)'
      % (len(y_target), DECODE_TARGET, y_target.min(), y_target.max()))

# contiguous folds, not shuffled: adjacent trials are correlated, and a shuffled
# split lets the model interpolate between neighbouring trials it has seen
cv = KFold(n_splits=N_FOLDS, shuffle=False)
ridge_pipeline = make_pipeline(StandardScaler(),
                               RidgeCV(alphas=np.logspace(-2, 4, 20)))

neural_scores = cross_val_score(ridge_pipeline, X_neural, y_target,
                                cv=cv, scoring='r2')
print('\nneural decode of %s: r2 = %.3f (folds %s)'
      % (DECODE_TARGET, neural_scores.mean(),
         np.array2string(neural_scores, precision=2)))

# shuffled control: the r2 a model achieves with the target relationship destroyed
rng = np.random.default_rng(0)
shuffle_scores = np.array([
    cross_val_score(ridge_pipeline, X_neural, rng.permutation(y_target),
                    cv=cv, scoring='r2').mean()
    for _ in range(20)])
print('shuffled target: r2 = %.3f +/- %.3f'
      % (shuffle_scores.mean(), shuffle_scores.std()))

# trial index alone: streak length may just track time in session
time_only_score = cross_val_score(
    ridge_pipeline, trial_position.reshape(-1, 1).astype(float), y_target,
    cv=cv, scoring='r2').mean()
print('trial index alone: r2 = %.3f' % time_only_score)

487 trials, target = consecutive_unrewarded (range 0-19)

neural decode of consecutive_unrewarded: r2 = -0.292 (folds [ 0.05  0.04 -1.51 -0.03 -0.01])


KeyboardInterrupt: 